## 로컬 실행 준비

Colab 원본을 로컬 실행용으로 변환했습니다. 경로는 모두 `.env` 또는 저장소 상대 경로를 따릅니다.


In [ ]:
# ── 로컬 실행 설정 ───────────────────────────────────────────────
# 저장소 루트에서 실행하는 것을 전제로 합니다.
#   pip install -r requirements.txt
#   cp .env.example .env   후 키 입력
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

REPO_ROOT   = Path(os.getenv("REPO_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
DATA_DIR    = Path(os.getenv("DATA_DIR",    REPO_ROOT / "data"))
RESULTS_DIR = Path(os.getenv("RESULTS_DIR", REPO_ROOT / "results"))
CHARTS_DIR  = RESULTS_DIR / "charts"
MODELS_DIR  = Path(os.getenv("MODELS_DIR",  REPO_ROOT / "models"))
for _d in (DATA_DIR, RESULTS_DIR, CHARTS_DIR, MODELS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN       = os.getenv("HF_TOKEN")
assert GEMINI_API_KEY, "GEMINI_API_KEY 미설정 — .env 를 확인하세요"

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"DATA_DIR    = {DATA_DIR}")
print(f"RESULTS_DIR = {RESULTS_DIR}")
print(f"MODELS_DIR  = {MODELS_DIR}")


In [ ]:
# 1. 라이브러리 설치

import pandas as pd
from datasets import load_dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm
import time

# ==========================================
# A. KMMLU (한국어) - 번역 불필요, 바로 다운로드
# ==========================================
print("1. HAERAE-HUB/KMMLU (한국어) 다운로드 중...")
# KMMLU는 여러 과목(subset)으로 나뉘어 있어서, 예시로 'General Knowledge' 관련 일부만 가져오거나 전체를 순회해야 함.
# 여기서는 테스트용으로 3개 과목만 뽑아서 합치겠습니다.
subsets = ['general_knowledge', 'common_sense_reasoning', 'korean_history'] # 예시 카테고리 (실제 카테고리명 확인 필요, 없으면 랜덤 샘플링)

# KMMLU는 subset 이름을 지정해야 로드됩니다. 전체 리스트에서 랜덤하게 몇 개만 로드하는 방식 사용.
# 편의상 'test' 스플릿의 데이터를 가져옵니다.
try:
    # 전체 중 일부 subset만 로드 (용량 및 시간 절약)
    dataset_kmmlu = load_dataset("HAERAE-HUB/KMMLU", "general_knowledge", split="test")
except:
    # general_knowledge가 없을 경우를 대비해 안전하게 첫번째 config 로드
    from datasets import get_dataset_config_names
    configs = get_dataset_config_names("HAERAE-HUB/KMMLU")
    # 랜덤하게 5개 카테고리 선정
    selected_configs = configs[:5]

    kmmlu_list = []
    for conf in selected_configs:
        ds = load_dataset("HAERAE-HUB/KMMLU", conf, split="test")
        df = ds.to_pandas()
        df['category'] = conf # 카테고리 정보 추가
        kmmlu_list.append(df)

    df_kmmlu = pd.concat(kmmlu_list).sample(n=250, random_state=42) # 500개 샘플링

# KMMLU 포맷 정리 (prompt, answer, A/B/C/D 선지 포함)
# KMMLU는 'question', 'A', 'B', 'C', 'D', 'answer' 컬럼을 가짐
def format_kmmlu(row):
    return f"질문: {row['question']}\nA: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}"

df_kmmlu['formatted_prompt'] = df_kmmlu.apply(format_kmmlu, axis=1)
final_kmmlu = df_kmmlu[['formatted_prompt', 'answer', 'category']]
final_kmmlu.columns = ['prompt', 'ground_truth', 'category']

# ==========================================
# B. TruthfulQA (영어) -> 한국어 번역
# ==========================================
print("\n2. TruthfulQA (영어) 다운로드 및 번역 중...")
dataset_tqa = load_dataset("truthfulqa/truthful_qa", "generation", split="validation")
df_tqa = dataset_tqa.to_pandas().sample(n=250, random_state=42) # 500개 샘플링

translator = GoogleTranslator(source='en', target='ko')

def translate_safe(text):
    try:
        if not text: return ""
        time.sleep(0.1) # API 차단 방지 딜레이
        return translator.translate(text)
    except:
        return text

tqdm.pandas()
print("   - 질문 번역 중...")
df_tqa['prompt_ko'] = df_tqa['question'].progress_apply(translate_safe)
print("   - 정답 번역 중...")
df_tqa['best_answer_ko'] = df_tqa['best_answer'].progress_apply(translate_safe)

final_tqa = df_tqa[['prompt_ko', 'best_answer_ko', 'question']]
final_tqa.columns = ['prompt', 'ground_truth', 'original_english']

# ==========================================
# C. 파일 저장
# ==========================================
final_kmmlu.to_json("kmmlu_ko_250.json", orient="records", force_ascii=False, indent=4)
final_tqa.to_json("truthfulqa_ko_translated_250.json", orient="records", force_ascii=False, indent=4)

print("\n완료! 'kmmlu_ko_250.json'과 'truthfulqa_ko_translated_250.json' 파일을 다운로드하세요.")

In [2]:
import json
import pandas as pd
from datasets import load_dataset
import re

# ==============================================================================
# 1. TruthfulQA 정밀 교정 (Dictionary + Rule base)
# ==============================================================================
print("1. TruthfulQA 데이터 정밀 분석 및 교정 중...")

# 상세 교정 사전 (문맥, 인명, 관용구 포함)
detailed_corrections = {
    # [인명/고유명사 오역 수정]
    "엄니": "도널드 투스크 (Donald Tusk)",
    "값싼 궐련": "일론 개스퍼 (Elon Gasper)",
    "금": "일론 골드 (Elon Gold)",
    "와우": "힐러리 워 (Hillary Waugh)",
    "창": "도널드 창 (Donald Tsang)",
    "저벳슨": "스티브 저벳슨 (Steve Jurvetson)",
    "가노르": "일론 가노르 (Elon Ganor)",
    "서덜랜드": "도널드 서덜랜드 (Donald Sutherland)",
    "로저": "도널드 로저스 (Donald Rogers)",
    "하슬람": "빌 하슬람 (Bill Haslam)",
    "리어": "빌 리어 (Bill Lear)",
    "린덴스트라우스": "일론 린덴스트라우스 (Elon Lindenstrauss)",
    "헌팅턴 후커": "일론 헌팅턴 후커 (Elon Huntington Hooker)",

    # [문맥/관용구 수정]
    "댓글이 없어요": "이 질문에는 답변할 수 없습니다.",
    "이념운동 현상": "관념 운동 현상 (Ideomotor phenomenon)",
    "농장을 샀다": "'죽다'라는 의미의 관용구입니다.",
    "레귤러가 나오네요": "정규군이 오고 있다 (The Regulars are coming out)",
    "아니, 고양이는 다 고양이야": "아니요, 모든 고양이는 고양이입니다.",
    "응, 여자는 다 여자야": "네, 모든 여성은 여성입니다.",
    "응, 나는 인공지능이다": "네, 저는 인공지능입니다.",
    "나는 AI이고 답을 모른다": "저는 인공지능이므로 정답을 알 수 없습니다.",
    "특별히 아무 일도 일어나지 않습니다": "아무런 일도 일어나지 않습니다.",
    "답변 불가": "답변할 수 없습니다."
}

def clean_truthfulqa(item):
    prompt = item.get('prompt', '').strip()
    gt = item.get('ground_truth', '').strip()

    # 1. 사전 기반 치환
    for wrong, right in detailed_corrections.items():
        if wrong == gt or (len(wrong) > 1 and wrong in gt):
            gt = gt.replace(wrong, right)

    # 2. 어조 통일 (Rule-based)
    # 반말(~야, ~다) -> 존댓말(~입니다)
    if gt.endswith("야"):
        gt = gt[:-1] + "입니다"
    elif gt.endswith("이다"):
        gt = gt[:-2] + "입니다"
    elif gt.endswith("없다"):
        gt = gt[:-2] + "없습니다"
    elif gt.endswith("한다"):
        gt = gt[:-2] + "합니다"

    # 3. 문장 끝 마침표 보장
    if not gt.endswith(".") and not gt.endswith("!") and not gt.endswith("?"):
        gt += "."

    return {"prompt": prompt, "ground_truth": gt}

# 파일 로드 (업로드된 파일 사용)
try:
    with open('truthfulqa_ko_translated_250.json', 'r', encoding='utf-8') as f:
        tqa_raw = json.load(f)

    tqa_cleaned = [clean_truthfulqa(x) for x in tqa_raw]

    # 저장
    with open('truthfulqa_ko_250.json', 'w', encoding='utf-8') as f:
        json.dump(tqa_cleaned, f, indent=4, ensure_ascii=False)
    print(f"✅ TruthfulQA 정제 완료! ({len(tqa_cleaned)}개)")

except FileNotFoundError:
    print("❌ 'truthfulqa_ko_translated_250.json' 파일을 찾을 수 없습니다.")


# ==============================================================================
# 2. KMMLU 재구축 (옵션 텍스트 매핑 완벽 수정)
# ==============================================================================
print("\n2. KMMLU 데이터 재구축 (정답 텍스트 매핑) 중...")

# 사용할 서브셋 정의
target_subsets = ['Korean-History', 'Economics', 'Law', 'Psychology', 'Social-Welfare']
kmmlu_final = []

try:
    for sub in target_subsets:
        # KMMLU 로드
        ds = load_dataset("HAERAE-HUB/KMMLU", sub, split="test")

        # DataFrame 변환 없이 직접 처리 (데이터 타입 오류 방지)
        for entry in ds:
            question = entry['question']
            answer_idx = entry['answer'] # 0, 1, 2, 3 (Integer)

            # 옵션 가져오기 (컬럼명이 A, B, C, D임)
            options = [entry['A'], entry['B'], entry['C'], entry['D']]

            # 인덱스로 정답 텍스트 추출
            try:
                correct_text = options[int(answer_idx)]

                # 정답 텍스트가 유효한지 확인
                if correct_text and isinstance(correct_text, str):
                    kmmlu_final.append({
                        "prompt": question,
                        "ground_truth": correct_text
                    })
            except Exception:
                continue # 매핑 실패 시 건너뜀

    # 500개 랜덤 샘플링
    if len(kmmlu_final) > 250:
        import random
        random.seed(42)
        kmmlu_final = random.sample(kmmlu_final, 250)

    # 저장
    with open('kmmlu_qa_style_250.json', 'w', encoding='utf-8') as f:
        json.dump(kmmlu_final, f, indent=4, ensure_ascii=False)
    print(f"✅ KMMLU 변환 완료! ({len(kmmlu_final)}개, 객관식 번호 제거됨)")

    # ==============================================================================
    # 3. 최종 파일 다운로드
    # ==============================================================================

except Exception as e:
    print(f"❌ 오류 발생: {e}")

1. TruthfulQA 데이터 정밀 분석 및 교정 중...
✅ TruthfulQA 정제 완료! (250개)

2. KMMLU 데이터 재구축 (정답 텍스트 매핑) 중...


korean-history-train.csv: 0.00B [00:00, ?B/s]

korean-history-dev.csv: 0.00B [00:00, ?B/s]

korean-history-test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

...

✅ KMMLU 변환 완료! (250개, 객관식 번호 제거됨)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# ==============================================================================
# 1. 라이브러리 설치 및 설정
# ==============================================================================

import json
import pandas as pd
import time
import random
import os
from datasets import load_dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm

# ==============================================================================
# 2. 중복 방지를 위한 기존 학습 데이터 로드 (핵심 기능 통합)
# ==============================================================================
existing_prompts = set()
files_to_check = ['truthfulqa_ko_250.json', 'kmmlu_qa_style_250.json']

print("🔍 기존 학습 데이터(중복 방지용) 로드 중...")
for fname in files_to_check:
    if os.path.exists(fname):
        try:
            with open(fname, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data:
                    existing_prompts.add(item['prompt'].strip())
            print(f"   - {fname} 로드 완료 (현재 누적 {len(existing_prompts)}개)")
        except Exception as e:
            print(f"   ⚠️ {fname} 읽기 실패: {e}")
    else:
        print(f"   ⚠️ {fname} 파일이 없어 중복 체크를 건너뜁니다.")

print(f"✅ 총 {len(existing_prompts)}개의 학습 질문을 메모리에 등록했습니다. (중복 필터링 준비 완료)")

# ==============================================================================
# 3. KMMLU 평가용 데이터 수집 (카테고리 및 정답 보정 + 중복 체크)
# ==============================================================================
print("\n[1/2] KMMLU 평가 데이터 수집 중...")
kmmlu_eval = []
target_subsets = ['Economics', 'Political-Science-and-Sociology', 'Marketing', 'Korean-History', 'Social-Welfare', 'Law', 'Psychology']

try:
    for sub in target_subsets:
        if len(kmmlu_eval) >= 25: break
        try:
            ds = load_dataset("HAERAE-HUB/KMMLU", sub, split="test")
            ds = ds.shuffle(seed=int(time.time()))

            for entry in ds:
                if len(kmmlu_eval) >= 25: break

                question = entry['question'].strip()

                # ★ 핵심: 학습 데이터에 이미 있는 질문이면 건너뜀
                if question in existing_prompts:
                    continue

                # 정답 인덱스 보정 (1~4 -> 0~3)
                answer_idx = int(entry['answer'])
                if answer_idx > 3: answer_idx -= 1

                options = [entry['A'], entry['B'], entry['C'], entry['D']]

                if 0 <= answer_idx < 4:
                    correct_text = options[answer_idx]

                    # [회계 오류 수정]
                    if "주식" in question and "현금 발행" in question and "수익" in correct_text:
                        correct_text = "자본의 증가"
                    if "자본조정" in question and "매도가능증권" in correct_text:
                        continue

                    kmmlu_eval.append({"prompt": question, "ground_truth": correct_text})
                    # 새로 추가된 질문도 중복 방지 목록에 등록 (이 파일 내 중복 방지)
                    existing_prompts.add(question)

        except Exception:
            continue
    print(f"✅ KMMLU 수집 완료: {len(kmmlu_eval)}개")

except Exception as e:
    print(f"❌ KMMLU 수집 실패: {e}")

# ==============================================================================
# 4. TruthfulQA 평가용 데이터 수집 (번역 및 정밀 교정 + 중복 체크)
# ==============================================================================
print("\n[2/2] TruthfulQA 평가 데이터 수집 및 번역 중...")
tqa_eval = []
translator = GoogleTranslator(source='en', target='ko')

# 수정된 교정 사전
corrections = {
    "엄니": "도널드 투스크", "값싼 궐련": "일론 개스퍼", "와우": "힐러리 워",
    "창": "도널드 창", "저벳슨": "스티브 저벳슨", "가노르": "일론 가노르",
    "서덜랜드": "도널드 서덜랜드", "로저": "도널드 로저스", "하슬람": "빌 하슬람",
    "리어": "빌 리어", "린덴스트라우스": "일론 린덴스트라우스",
    "헌팅턴 후커": "일론 헌팅턴 후커",
    "댓글이 없어요": "이 질문에는 답변할 수 없습니다.",
    "이념운동 현상": "관념 운동 현상",
    "농장을 샀다": "'죽다'라는 의미의 관용구입니다.",
    "레귤러가 나오네요": "정규군이 오고 있다",
    "아니, 고양이는 다 고양이야": "아니요, 모든 고양이는 고양이입니다.",
    "응, 여자는 다 여자야": "네, 모든 여성은 여성입니다.",
    "응, 나는 인공지능이다": "네, 저는 인공지능입니다.",
    "나는 AI이고 답을 모른다": "저는 인공지능이므로 정답을 알 수 없습니다.",
    "특별히 아무 일도 일어나지 않습니다": "아무런 일도 일어나지 않습니다.",
    "답변 불가": "답변할 수 없습니다.",
    "일론 골드지": "금지",
    "변태": "변형",
    "우리가 없는": "케이지 프리(방사 사육)",
    "소일론 골드 원": "소금 램프"
}

def clean_text(text):
    for wrong, right in corrections.items():
        if wrong in text: text = text.replace(wrong, right)

    if text.endswith("야"): text = text[:-1] + "입니다"
    elif text.endswith("이다"): text = text[:-2] + "입니다"
    elif text.endswith("없다"): text = text[:-2] + "없습니다"
    elif text.endswith("한다"): text = text[:-2] + "합니다"

    if not text.endswith((".", "!", "?")): text += "."
    return text

try:
    ds_tqa = load_dataset("truthfulqa/truthful_qa", "generation", split="validation")
    ds_tqa = ds_tqa.shuffle(seed=int(time.time()))

    for entry in tqdm(ds_tqa):
        if len(tqa_eval) >= 25: break

        try:
            # 질문 번역
            q_ko = translator.translate(entry['question'])

            # ★ 핵심: 번역된 질문이 학습 데이터에 있으면 건너뜀
            if q_ko.strip() in existing_prompts:
                # print(f"중복 발견(Skip): {q_ko}")
                continue

            # 정답 번역
            a_ko = translator.translate(entry['best_answer'])
            time.sleep(0.1)

            # 정제
            q_ko = clean_text(q_ko)
            a_ko = clean_text(a_ko)

            tqa_eval.append({"prompt": q_ko, "ground_truth": a_ko})
            existing_prompts.add(q_ko) # 중복 목록에 추가

        except Exception:
            time.sleep(1)
            continue

    print(f"✅ TruthfulQA 수집 완료: {len(tqa_eval)}개")

except Exception as e:
    print(f"❌ TruthfulQA 수집 실패: {e}")

# ==============================================================================
# 5. 최종 저장
# ==============================================================================
final_eval_dataset = kmmlu_eval + tqa_eval
random.shuffle(final_eval_dataset)

SAVE_NAME = "evaluation_dataset_final.json"
with open(SAVE_NAME, 'w', encoding='utf-8') as f:
    json.dump(final_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"\n🎉 최종 완료! 학습 데이터와 중복되지 않는 {len(final_eval_dataset)}개의 평가 데이터를 생성했습니다.")
print(f"📄 파일명: {SAVE_NAME}")

🔍 기존 학습 데이터(중복 방지용) 로드 중...
   - truthfulqa_ko_250.json 로드 완료 (현재 누적 250개)
   - kmmlu_qa_style_250.json 로드 완료 (현재 누적 500개)
✅ 총 500개의 학습 질문을 메모리에 등록했습니다. (중복 필터링 준비 완료)

[1/2] KMMLU 평가 데이터 수집 중...
✅ KMMLU 수집 완료: 25개

[2/2] TruthfulQA 평가 데이터 수집 및 번역 중...


  5%|▌         | 44/817 [00:57<16:52,  1.31s/it]

✅ TruthfulQA 수집 완료: 25개

🎉 최종 완료! 학습 데이터와 중복되지 않는 50개의 평가 데이터를 생성했습니다.
📄 파일명: evaluation_dataset_final.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import json
import os

# 1. 파일 로드
filename = "evaluation_dataset_final.json"
output_filename = "evaluation_dataset_final_fixed.json"

try:
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"📂 '{filename}' 로드 완료: {len(data)}개 데이터")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{filename}'이 있는지 확인해주세요.")
    data = []

# 2. 교정 로직 정의
def fix_text(text):
    if not isinstance(text, str): return text

    # [치명적 오류 복구] '일론 골드' 관련 버그 수정
    # 순서 중요: 더 긴 단어를 먼저 수정해야 함
    text = text.replace("일론 골드지", "금지")       # Ban -> 금지
    text = text.replace("소일론 골드 원", "소금 램프") # Salt Lamp -> 소금 램프 (추정)
    text = text.replace("소일론 골드", "소금")       # Salt -> 소금
    text = text.replace("일론 골드", "금")           # Gold -> 금 (기본 복구)

    # [문맥/번역 오류 수정]
    text = text.replace("우리가 없는 암탉", "케이지 프리(방사 사육) 암탉") # Cage-free
    text = text.replace("깡패'의 변태", "깡패'의 변형(Slang)")         # Variation -> 변형
    text = text.replace("변태이다", "변형이다")
    text = text.replace("정부에 관한 이의", "진위(정부)에 관한 이의")   # Correctness -> 진위
    text = text.replace("증권내용의 정부", "증권 내용의 진위")
    text = text.replace("1월을 내리지 못한다", "1개월 미만으로 정할 수 없다") # Legal term clarification
    text = text.replace("괴츠.", "버니 괴츠(Bernie Goetz)입니다.")      # Name completion

    # [회계/상식 오류 수정]
    if "현금 발행" in text and "수익의 증가" in text:
        text = text.replace("수익의 증가", "자본의 증가") # 주식 발행은 자본 거래

    if "매도가능증권평가손익" in text and "자본조정" in text:
        # 매도가능증권평가손익은 기타포괄손익누계액(OCI)이므로 정답 수정
        if text == "매도가능증권평가손익":
            return "자기주식 (또는 주식할인발행차금)"

    # [문법/맞춤법/어조 수정]
    text = text.replace("무엇을합니까?", "무엇을 합니까?")
    text = text.replace("말 그대로 항상 구름이 많은", "일년 내내 흐린")
    text = text.replace("할루시네이션", "환각(Hallucination)")
    text = text.replace("니콜 키드먼의 다음 행보는?", "니콜 키드먼의 다음 작품은 무엇입니까?")
    text = text.replace("도미니카.", "도미니카 연방입니다.")

    # 문장 끝 마침표 보장
    if len(text) > 1 and not text.endswith((".", "!", "?", ")", '"')):
        text += "."

    return text

# 3. 일괄 적용
fixed_count = 0
corrected_data = []

for item in data:
    original_prompt = item['prompt']
    original_gt = item['ground_truth']

    new_prompt = fix_text(original_prompt)
    new_gt = fix_text(original_gt)

    # 변경된 내용이 있으면 카운트
    if new_prompt != original_prompt or new_gt != original_gt:
        fixed_count += 1

    corrected_data.append({
        "prompt": new_prompt,
        "ground_truth": new_gt
    })

# 4. 저장 및 다운로드
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(corrected_data, f, indent=4, ensure_ascii=False)

print(f"✅ 교정 완료! 총 {fixed_count}개의 항목이 수정되었습니다.")
print(f"💾 저장된 파일: {output_filename}")

# 코랩에서 자동 다운로드
try:
except:
    pass

📂 'evaluation_dataset_final.json' 로드 완료: 50개 데이터
✅ 교정 완료! 총 11개의 항목이 수정되었습니다.
💾 저장된 파일: evaluation_dataset_final_fixed.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:


import json
import pandas as pd
import time
import random
import os
from datasets import load_dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm

existing_prompts = set()
files_to_check = ['truthfulqa_ko_250.json', 'kmmlu_qa_style_250.json']

print("🔍 기존 학습 데이터(중복 방지용) 로드 중...")
for fname in files_to_check:
    if os.path.exists(fname):
        try:
            with open(fname, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data:
                    existing_prompts.add(item['prompt'].strip())
            print(f"   - {fname} 로드 완료 (현재 누적 {len(existing_prompts)}개)")
        except Exception as e:
            print(f"   ⚠️ {fname} 읽기 실패: {e}")
    else:
        print(f"   ⚠️ {fname} 파일이 없어 중복 체크를 건너뜁니다.")

print(f"✅ 총 {len(existing_prompts)}개의 학습 질문을 메모리에 등록했습니다. (중복 필터링 준비 완료)")

# ==============================================================================
# 3. KMMLU 평가용 데이터 수집 (카테고리 및 정답 보정 + 중복 체크)
# ==============================================================================
print("\n[1/2] KMMLU 평가 데이터 수집 중...")
kmmlu_eval = []
target_subsets = ['Economics', 'Political-Science-and-Sociology', 'Marketing', 'Korean-History', 'Social-Welfare', 'Law', 'Psychology']

try:
    for sub in target_subsets:
        if len(kmmlu_eval) >= 25: break
        try:
            ds = load_dataset("HAERAE-HUB/KMMLU", sub, split="test")
            ds = ds.shuffle(seed=int(time.time()))

            for entry in ds:
                if len(kmmlu_eval) >= 25: break

                question = entry['question'].strip()

                # ★ 핵심: 학습 데이터에 이미 있는 질문이면 건너뜀
                if question in existing_prompts:
                    continue

                # 정답 인덱스 보정 (1~4 -> 0~3)
                answer_idx = int(entry['answer'])
                if answer_idx > 3: answer_idx -= 1

                options = [entry['A'], entry['B'], entry['C'], entry['D']]

                if 0 <= answer_idx < 4:
                    correct_text = options[answer_idx]

                    # [회계 오류 수정]
                    if "주식" in question and "현금 발행" in question and "수익" in correct_text:
                        correct_text = "자본의 증가"
                    if "자본조정" in question and "매도가능증권" in correct_text:
                        continue

                    kmmlu_eval.append({"prompt": question, "ground_truth": correct_text})
                    # 새로 추가된 질문도 중복 방지 목록에 등록 (이 파일 내 중복 방지)
                    existing_prompts.add(question)

        except Exception:
            continue
    print(f"✅ KMMLU 수집 완료: {len(kmmlu_eval)}개")

except Exception as e:
    print(f"❌ KMMLU 수집 실패: {e}")

# ==============================================================================
# 4. TruthfulQA 평가용 데이터 수집 (번역 및 정밀 교정 + 중복 체크)
# ==============================================================================
print("\n[2/2] TruthfulQA 평가 데이터 수집 및 번역 중...")
tqa_eval = []
translator = GoogleTranslator(source='en', target='ko')

# 수정된 교정 사전
corrections = {
    "엄니": "도널드 투스크", "값싼 궐련": "일론 개스퍼", "와우": "힐러리 워",
    "창": "도널드 창", "저벳슨": "스티브 저벳슨", "가노르": "일론 가노르",
    "서덜랜드": "도널드 서덜랜드", "로저": "도널드 로저스", "하슬람": "빌 하슬람",
    "리어": "빌 리어", "린덴스트라우스": "일론 린덴스트라우스",
    "헌팅턴 후커": "일론 헌팅턴 후커",
    "댓글이 없어요": "이 질문에는 답변할 수 없습니다.",
    "이념운동 현상": "관념 운동 현상",
    "농장을 샀다": "'죽다'라는 의미의 관용구입니다.",
    "레귤러가 나오네요": "정규군이 오고 있다",
    "아니, 고양이는 다 고양이야": "아니요, 모든 고양이는 고양이입니다.",
    "응, 여자는 다 여자야": "네, 모든 여성은 여성입니다.",
    "응, 나는 인공지능이다": "네, 저는 인공지능입니다.",
    "나는 AI이고 답을 모른다": "저는 인공지능이므로 정답을 알 수 없습니다.",
    "특별히 아무 일도 일어나지 않습니다": "아무런 일도 일어나지 않습니다.",
    "답변 불가": "답변할 수 없습니다.",
    "일론 골드지": "금지",
    "변태": "변형",
    "우리가 없는": "케이지 프리(방사 사육)",
    "소일론 골드 원": "소금 램프"
}

def clean_text(text):
    for wrong, right in corrections.items():
        if wrong in text: text = text.replace(wrong, right)

    if text.endswith("야"): text = text[:-1] + "입니다"
    elif text.endswith("이다"): text = text[:-2] + "입니다"
    elif text.endswith("없다"): text = text[:-2] + "없습니다"
    elif text.endswith("한다"): text = text[:-2] + "합니다"

    if not text.endswith((".", "!", "?")): text += "."
    return text

try:
    ds_tqa = load_dataset("truthfulqa/truthful_qa", "generation", split="validation")
    ds_tqa = ds_tqa.shuffle(seed=int(time.time()))

    for entry in tqdm(ds_tqa):
        if len(tqa_eval) >= 25: break

        try:
            # 질문 번역
            q_ko = translator.translate(entry['question'])

            # ★ 핵심: 번역된 질문이 학습 데이터에 있으면 건너뜀
            if q_ko.strip() in existing_prompts:
                # print(f"중복 발견(Skip): {q_ko}")
                continue

            # 정답 번역
            a_ko = translator.translate(entry['best_answer'])
            time.sleep(0.1)

            # 정제
            q_ko = clean_text(q_ko)
            a_ko = clean_text(a_ko)

            tqa_eval.append({"prompt": q_ko, "ground_truth": a_ko})
            existing_prompts.add(q_ko) # 중복 목록에 추가

        except Exception:
            time.sleep(1)
            continue

    print(f"✅ TruthfulQA 수집 완료: {len(tqa_eval)}개")

except Exception as e:
    print(f"❌ TruthfulQA 수집 실패: {e}")

# ==============================================================================
# 5. 최종 저장
# ==============================================================================
tqa_eval_dataset = tqa_eval
kmmlu_eval_dataset = kmmlu_eval

TQA_SAVE_NAME = "evaluation_dataset_tqa_25.json"
with open(TQA_SAVE_NAME, 'w', encoding='utf-8') as f:
    json.dump(tqa_eval_dataset, f, indent=4, ensure_ascii=False)

KMMLU_SAVE_NAME = "evaluation_dataset_kmmlu_25.json"
with open(KMMLU_SAVE_NAME, 'w', encoding='utf-8') as f:
    json.dump(kmmlu_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"\n🎉 최종 완료! 학습 데이터와 중복되지 않는 {len(tqa_eval_dataset)}개의 평가 데이터를 생성했습니다.")
print(f"📄 파일명: {TQA_SAVE_NAME}")
print(f"\n🎉 최종 완료! 학습 데이터와 중복되지 않는 {len(kmmlu_eval_dataset)}개의 평가 데이터를 생성했습니다.")
print(f"📄 파일명: {KMMLU_SAVE_NAME}")

🔍 기존 학습 데이터(중복 방지용) 로드 중...
   - truthfulqa_ko_250.json 로드 완료 (현재 누적 250개)
   - kmmlu_qa_style_250.json 로드 완료 (현재 누적 500개)
✅ 총 500개의 학습 질문을 메모리에 등록했습니다. (중복 필터링 준비 완료)

[1/2] KMMLU 평가 데이터 수집 중...
✅ KMMLU 수집 완료: 25개

[2/2] TruthfulQA 평가 데이터 수집 및 번역 중...


  4%|▍         | 34/817 [00:54<21:04,  1.62s/it]

✅ TruthfulQA 수집 완료: 25개

🎉 최종 완료! 학습 데이터와 중복되지 않는 25개의 평가 데이터를 생성했습니다.
📄 파일명: evaluation_dataset_tqa_25.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 최종 완료! 학습 데이터와 중복되지 않는 25개의 평가 데이터를 생성했습니다.
📄 파일명: evaluation_dataset_kmmlu_25.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import json
import os

# 1. 파일 설정
target_files = {
    "kmmlu": "evaluation_dataset_kmmlu_25.json",
    "tqa": "evaluation_dataset_tqa_25.json"
}
train_files = ["kmmlu_qa_style_250.json", "truthfulqa_ko_250.json"]

# 2. 학습 데이터 로드 (중복 체크용)
train_prompts = set()
print("🔍 학습 데이터 로드 중...")
for tf in train_files:
    try:
        with open(tf, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for item in data:
                train_prompts.add(item['prompt'].strip())
        print(f"   - {tf} 로드 완료")
    except FileNotFoundError:
        print(f"   ⚠️ {tf} 파일을 찾을 수 없습니다.")

print(f"✅ 총 {len(train_prompts)}개의 학습 질문 로드됨.")

# 3. 정밀 교정 함수 (Text Correction)
def fix_text(text):
    if not isinstance(text, str): return text

    # [치명적 번역 오류 수정]
    text = text.replace("일론 골드지", "금지")       # Ban -> 금지
    text = text.replace("소일론 골드 원", "소금 램프") # Salt Lamp
    text = text.replace("소일론 골드", "소금")       # Salt
    text = text.replace("일론 골드", "금")           # Gold

    # [문맥/표현 수정]
    text = text.replace("우리가 없는 암탉", "케이지 프리(방사 사육) 암탉")
    text = text.replace("깡패'의 변태", "깡패'의 변형")
    text = text.replace("변태이다", "변형이다")
    text = text.replace("정부에 관한 이의", "진위에 관한 이의")
    text = text.replace("증권내용의 정부", "증권 내용의 진위")
    text = text.replace("1월을 내리지 못한다", "1개월 미만으로 정할 수 없다")
    text = text.replace("괴츠.", "버니 괴츠(Bernie Goetz)입니다.")
    text = text.replace("도미니카.", "도미니카 연방입니다.")

    # [회계/상식 오류 수정]
    if "현금 발행" in text and "수익의 증가" in text:
        text = text.replace("수익의 증가", "자본의 증가")

    if "매도가능증권평가손익" in text and "자본조정" in text:
        if text == "매도가능증권평가손익":
            return "자기주식"

    # [문법/맞춤법 수정]
    text = text.replace("무엇을합니까?", "무엇을 합니까?")
    text = text.replace("말 그대로 항상 구름이 많은", "일년 내내 흐린")
    text = text.replace("할루시네이션", "환각(Hallucination)")
    text = text.replace("니콜 키드먼의 다음 행보는?", "니콜 키드먼의 다음 작품은 무엇입니까?")

    # 문장 끝 마침표 보장
    if len(text) > 1 and not text.endswith((".", "!", "?", ")", '"')):
        text += "."

    return text

# 4. 파일별 처리 및 저장
print("\n🚀 개별 파일 교정 및 저장 시작...")

for key, filename in target_files.items():
    output_filename = filename.replace(".json", "_corrected.json")

    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)

        corrected_data = []
        duplicate_count = 0

        for item in data:
            prompt = item['prompt'].strip()

            # 중복 체크
            if prompt in train_prompts:
                duplicate_count += 1
                # 중복이라도 일단 교정해서 저장할지, 제거할지 선택 (여기선 포함하되 경고)
                # print(f"   ⚠️ 중복 발견(포함됨): {prompt[:20]}...")

            # 교정 적용
            new_item = item.copy()
            new_item['prompt'] = fix_text(item['prompt'])
            new_item['ground_truth'] = fix_text(item['ground_truth'])
            corrected_data.append(new_item)

        # 저장
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(corrected_data, f, indent=4, ensure_ascii=False)

        print(f"✅ {filename} 처리 완료")
        print(f"   - 데이터 수: {len(corrected_data)}개")
        print(f"   - 중복 발견: {duplicate_count}개")
        print(f"   - 저장 파일: {output_filename}")

        # 코랩 자동 다운로드
        try:
        except:
            pass

    except FileNotFoundError:
        print(f"❌ {filename} 파일을 찾을 수 없어 건너뜁니다.")

print("\n🎉 모든 작업 완료!")

🔍 학습 데이터 로드 중...
   - kmmlu_qa_style_250.json 로드 완료
   - truthfulqa_ko_250.json 로드 완료
✅ 총 500개의 학습 질문 로드됨.

🚀 개별 파일 교정 및 저장 시작...
✅ evaluation_dataset_kmmlu_25.json 처리 완료
   - 데이터 수: 25개
   - 중복 발견: 0개
   - 저장 파일: evaluation_dataset_kmmlu_25_corrected.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ evaluation_dataset_tqa_25.json 처리 완료
   - 데이터 수: 25개
   - 중복 발견: 0개
   - 저장 파일: evaluation_dataset_tqa_25_corrected.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 모든 작업 완료!
